# Chapter 2: Seaborn — Statistical Graphics With Minimal Code

### Why This Matters

Matplotlib asks "where exactly should this line go?" Seaborn asks "what's the relationship in this data?" and handles the plotting mechanics for you — grouping by category, computing confidence intervals, choosing sensible colors — from a single line pointed at a DataFrame. For anything involving *comparing groups* or *understanding distributions*, Seaborn will get you to a good-looking, statistically sensible chart faster than hand-rolled Matplotlib.

### 2.1 The Core Seaborn Chart Types

In [ ]:
np.random.seed(42)
df = pd.DataFrame({
    "Region": np.random.choice(["North", "South", "East", "West"], 200),
    "Sales": np.random.normal(120, 30, 200),
    "Profit_Margin": np.random.normal(18, 5, 200),
    "Customer_Type": np.random.choice(["New", "Returning"], 200)
})

# Distribution of a single numeric column
sns.histplot(df["Sales"], bins=20, kde=True)
plt.title("Distribution of Sales", fontsize=14, fontweight='bold')
plt.show()

`kde=True` overlays a smooth **Kernel Density Estimate** curve on top of the histogram bars — essentially a smoothed version of the same distribution, useful for seeing the overall shape (is it symmetric? skewed? bimodal?) without the visual noise of individual bin heights. This is precisely the kind of "shape of the data" question you'll formalize statistically in Volume 04.

In [ ]:
# Comparing a numeric variable ACROSS categories — one of the single most useful business chart types
sns.boxplot(data=df, x="Region", y="Sales")
plt.title("Sales Distribution by Region", fontsize=14, fontweight='bold')
plt.show()

A **box plot** packs five numbers into one visual per category: the median (the line inside the box), the 25th/75th percentiles (the box edges — together, the "interquartile range" you'll formalize in Volume 04), and the overall spread including outliers (the whiskers and any individual dots beyond them). Where a bar chart of *averages* can hide the fact that one region has wildly inconsistent performance while another is steady, a box plot surfaces that spread immediately — which is exactly the kind of nuance an MBA-trained reader is trained to look for and should learn to build into charts by default.

In [ ]:
# Grouped bar comparison — two categorical dimensions at once
sns.barplot(data=df, x="Region", y="Sales", hue="Customer_Type", errorbar=None)
plt.title("Average Sales by Region & Customer Type", fontsize=14, fontweight='bold')
plt.show()

`hue=` is the single feature that makes Seaborn dramatically more efficient than hand-building the equivalent in Matplotlib: it automatically splits the bars (or points, or lines) by a second categorical column and color-codes them with an automatic, readable legend — no manual looping over subgroups required. `errorbar=None` turns off Seaborn's default confidence-interval whiskers on the bars (shown by default because `barplot` implicitly treats each bar as an *estimate* of a population mean) — worth turning off for a cleaner executive chart when you don't need to communicate statistical uncertainty, and worth *keeping on* when you do (e.g., comparing two small samples where the difference might not be statistically meaningful — see Volume 04's hypothesis testing chapter).

In [ ]:
# Relationship between two numeric variables
sns.scatterplot(data=df, x="Sales", y="Profit_Margin", hue="Region", s=80, alpha=0.7)
plt.title("Sales vs Profit Margin by Region", fontsize=14, fontweight='bold')
plt.show()

`alpha=0.7` (partial transparency) is a small but valuable habit for scatter plots with any real number of points — where points overlap, the overlap becomes visually darker, giving you a rough sense of density for free, which a fully opaque scatter plot hides.

### 2.2 Regression and Faceted Views

In [ ]:
df['visits'] = df['Sales']/20 + np.random.normal(0, 3, len(df))

sns.regplot(data=df, x='visits', y='Sales')
plt.title("Visits vs. Sales, with Fitted Trend Line")
plt.show()

`sns.regplot` overlays a fitted linear regression line (plus its confidence band, shaded) directly on a scatter plot in one call — a fast, visual first check of "is there a linear relationship here, and how strong/noisy is it?" before you commit to building a formal regression model in Volume 04–05.

💡 One more Seaborn capability worth knowing exists even though the source notebooks only gesture at it: **`sns.FacetGrid` / `sns.relplot(..., col="category")`** lets you automatically generate a *grid* of small charts, one per category (e.g., one scatter plot per region, side by side) — extremely useful for "does this relationship hold consistently across every segment, or only in some?" questions, without manually looping and building subplots yourself.

### Cheat Sheet — Seaborn

| Question | Chart | Function |
|---|---|---|
| Shape of one numeric variable | Histogram + density | `sns.histplot(x, kde=True)` |
| Compare a numeric variable across categories | Box plot | `sns.boxplot(data, x=cat, y=num)` |
| Compare category averages, split by a 2nd category | Grouped bar | `sns.barplot(data, x, y, hue=)` |
| Relationship between two numeric variables | Scatter | `sns.scatterplot(data, x, y, hue=)` |
| ...plus a fitted trend line | Regression plot | `sns.regplot(data, x, y)` |
| Same chart repeated per category | Facet grid | `sns.relplot(data, x, y, col=cat)` |

### 🎯 Business Challenge — Seaborn

*Task (from the source material's "mini project," expanded):* produce a five-chart customer analytics view with a one-sentence takeaway under each.

In [ ]:
np.random.seed(7)
customers = pd.DataFrame({
    "segment": np.random.choice(["Value", "Core", "Premium"], 600),
    "spend": np.random.gamma(4, 50, 600)
})
customers["visits"] = customers["spend"]/20 + np.random.normal(0, 3, len(customers))

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
sns.histplot(data=customers, x="spend", hue="segment", kde=True, ax=axes[0,0])
axes[0,0].set_title("Spend distribution by segment")
sns.boxplot(data=customers, x="segment", y="spend", ax=axes[0,1])
axes[0,1].set_title("Spend spread by segment")
sns.scatterplot(data=customers, x="visits", y="spend", hue="segment", alpha=0.6, ax=axes[0,2])
axes[0,2].set_title("Visits vs. spend")
sns.barplot(data=customers, x="segment", y="spend", errorbar="sd", ax=axes[1,0])
axes[1,0].set_title("Average spend +/- 1 std dev")
sns.violinplot(data=customers, x="segment", y="spend", ax=axes[1,1])
axes[1,1].set_title("Full distribution shape by segment")
axes[1,2].axis("off")     # leave the last panel empty, or use it for a text summary
plt.tight_layout()
plt.show()

*Business sentence for panel 1:* "Premium customers show the widest spread in spend, suggesting the segment isn't as homogeneous as the label implies — worth sub-segmenting further."

---